# Physics-Informed Neural Network (PINN) Battery Surrogate Training & Ablation

This notebook demonstrates the training and evaluation of the **Battery PINN Surrogate** for the EcoGrid-Twin microgrid.

### Objectives:
1. Train a Physics-Informed Neural Network (`BatteryPINNNet`, 5 features: `[soc_prev, temp_prev, actual_power_kw, ambient_temp_c, dt_hours]`) enforcing electrochemical energy conservation and lumped thermal dynamics.
2. Compare PINN against an identical Plain-MLP ablation across data-scarcity regimes (1%, 5%, 10%, 100% training trajectories).
3. Evaluate decoupled domain-wide collocation residuals and safe operating boundary violations [15%, 95%].
4. Contrast against deterministic analytical Thevenin ODE baselines and verify integration with `BatteryPINN.estimate_state(...)`.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Change directory to project root if running from notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working Directory:', os.getcwd())

## 1. Inspect Battery Trajectory Dataset
The dataset contains multi-day simulation trajectories generated using the physical Thevenin battery model.

In [ ]:
df = pd.read_csv('dataset/battery_sim_trajectories.csv', comment='#')
print(f'Total records: {len(df)}, Trajectories: {df["trajectory_id"].nunique()}')
df.head()

## 2. Train PINN and Run Ablation Study
Executes `backend.ai.pinn.train.train_pinn()` to train both the PINN and the unconstrained Plain-MLP baseline.

In [ ]:
from backend.ai.pinn.train import train_pinn

metrics = train_pinn()
print(json.dumps(metrics, indent=2))

## 3. Visualize Ablation: PINN vs Plain MLP

In [ ]:
img_path = 'output_figures/fig_pinn_ablation.png'
if os.path.exists(img_path):
    img = Image.open(img_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('PINN vs Plain MLP Ablation: Extrapolation & Physical Consistency', fontsize=14)
    plt.show()
else:
    print('Ablation figure not found.')

## 4. Test Live Inference with `BatteryPINN`
Verify that `BatteryPINN.estimate_state(...)` uses the trained neural surrogate.

In [ ]:
from backend.ai.pinn.model import BatteryPINN

pinn = BatteryPINN(capacity_kwh=250.0, efficiency=0.92)

# Discharge test: 45 kW discharge for 15 min at 30°C
res_dis = pinn.estimate_state(current_soc=75.0, target_power_kw=45.0, dt_hours=0.25, ambient_temp=30.0)
print('Discharge Result:', res_dis)

# Charge test: -40 kW charge for 15 min at 28°C
res_chg = pinn.estimate_state(current_soc=45.0, target_power_kw=-40.0, dt_hours=0.25, ambient_temp=28.0)
print('Charge Result:   ', res_chg)